# CDI Credit Risk Model — Challenger (v2)

Self-contained demo with synthetic data, base-case simulation, stress scenarios, and ICAAP.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from pathlib import Path
import sys, importlib

sys.path.insert(0, str(Path.cwd()))

# Auto-reload on every cell
def _reload():
    if 'model_v2' in sys.modules:
        importlib.reload(sys.modules['model_v2'])
from IPython.core.getipython import get_ipython
ip = get_ipython()
if ip:
    ip.events.register('pre_execute', _reload)

from model_v2 import *

## 1 — Synthetic data generators

In [ ]:
# ── Rating schema ────────────────────────────────────────────────────
schema = DEFAULT_SCHEMA

# ── Transition matrix (stylised) ─────────────────────────────────────
n_states = schema.n
raw = np.zeros((n_states, n_states))
for i in range(n_states):
    if i == 0:                               # AAAA absorbing
        raw[i, i] = 1.0
    elif i == n_states - 1:                   # Default absorbing
        raw[i, i] = 1.0
    else:
        stay   = max(0.80, 0.97 - 0.01 * i)
        up     = min(0.05, (0.97 - stay) * 0.3)
        down   = min(0.08, (0.97 - stay) * 0.5)
        to_def = max(0.0005, 0.0001 * (1.4 ** i))
        resid  = 1.0 - stay - up - down - to_def
        if resid < 0:
            stay += resid
            resid = 0
        raw[i, max(0, i - 1)] += up
        raw[i, i] += stay
        raw[i, min(n_states - 2, i + 1)] += down + resid
        raw[i, n_states - 1] += to_def

# Ensure rows sum to 1
raw /= raw.sum(axis=1, keepdims=True)
tmat = TransitionMatrix(raw, schema)

print(f'States: {schema.n}  |  Default idx: {schema.default_idx}')
print(f'Time to default (BBB): {tmat.expected_time_to_default()["BBB"]:.0f} years')

In [ ]:
# ── Valuation date & horizon ─────────────────────────────────────────
val_date = pd.Timestamp('2026-01-01')
n_years  = 25
dates    = pd.date_range(val_date, periods=n_years, freq='YE')

# ── Yield curve (German Bund proxy) ──────────────────────────────────
tenors = np.arange(1, n_years + 1, dtype=float)
bund_y = 0.025 + 0.005 * np.log(tenors)           # ~2.5% short, ~3.1% long
curve  = YieldCurve(bund_y, dates)

# ── Spread table (indexed by rating int) ─────────────────────────────
base_spreads = np.array([
    0.0000,  # AAAA
    0.0020,  # AAA
    0.0030,  # AA+
    0.0035,  # AA
    0.0040,  # AA-
    0.0050,  # A+
    0.0060,  # A
    0.0075,  # A-
    0.0095,  # BBB+
    0.0120,  # BBB
    0.0155,  # BBB-
    0.0200,  # BB+
    0.0260,  # BB
    0.0340,  # BB-
    0.0430,  # B+
    0.0550,  # B
    0.0700,  # B-
    0.0900,  # CCC+
    0.1100,  # CCC
    0.1400,  # CCC-
    0.1800,  # CC+
    0.0000,  # Def (irrelevant)
])

# ── Issuers ──────────────────────────────────────────────────────────
n_issuers = 40
rng_data  = np.random.default_rng(42)
issuer_ratings_str = rng_data.choice(
    ['AA', 'AA-', 'A+', 'A', 'A-', 'BBB+', 'BBB', 'BBB-'],
    size=n_issuers,
    p=[0.05, 0.10, 0.15, 0.20, 0.15, 0.15, 0.12, 0.08],
)
issuer_rating_idx = schema.to_idx(issuer_ratings_str)
sector_ids = rng_data.integers(0, 1, size=n_issuers)   # single sector

# ── Bonds (1 bond per issuer, bullet + coupon) ───────────────────────
n_bonds   = n_issuers
issuer_of = np.arange(n_bonds)                          # bond i → issuer i
maturities = rng_data.integers(5, 26, size=n_bonds)     # 5 – 25 yr
coupons    = bund_y[0] + base_spreads[issuer_rating_idx] + rng_data.uniform(-0.002, 0.002, n_bonds)
recoveries = np.full(n_bonds, 0.40)

# Build cashflow matrix (n_years × n_bonds)
cf_matrix = np.zeros((n_years, n_bonds))
for b in range(n_bonds):
    mat = min(maturities[b], n_years)
    cf_matrix[:mat, b] = coupons[b]                      # coupon
    cf_matrix[mat - 1, b] += 1.0                         # principal

bonds = BondUniverse(
    cashflows=cf_matrix, dates=dates,
    issuer_ids=issuer_of, recoveries=recoveries,
    spread_table=base_spreads,
)

# ── Allocations (notional per bond) ──────────────────────────────────
cdi_w  = rng_data.uniform(3e6, 12e6, n_bonds)
cmbp_w = np.zeros(n_bonds)
# First 5 bonds are CMBP (AAAA-rated government)
cmbp_idx = np.arange(5)
issuer_rating_idx[cmbp_idx] = 0                          # AAAA
cmbp_w[cmbp_idx] = rng_data.uniform(5e6, 15e6, len(cmbp_idx))

# ── Liabilities ──────────────────────────────────────────────────────
liab_cf = 12e6 * np.exp(-0.02 * np.arange(n_years))    # declining ~12M → ~7M
liabs   = Liabilities(cashflows=liab_cf, dates=dates)

# ── Mandate config ───────────────────────────────────────────────────
cfg = MandateConfig(
    heubeck_liabilities=297e6,
    r_gaap=0.0201, r_ifrs=0.039,
    mortality_buffer=1.0559, cmbp_margin=0.001,
    fee_rate=0.003, asset_buffer_0=12.5e6,
    performance_cap=50e6,
)

# ── CDI mandate object ───────────────────────────────────────────────
mandate = CDIMandate(
    bonds=bonds, liabs=liabs, cfg=cfg,
    cdi_weights=cdi_w, cmbp_weights=cmbp_w,
    cash_0=2e6,
)

# ── Credit model ─────────────────────────────────────────────────────
rho_e = 0.24
rho_s = np.array([rho_e])                                # single sector

cr_model = CreditModel(
    tmatrix=tmat, rho_e=rho_e, rho_s=rho_s,
    n_issuers=n_issuers, sector_ids=sector_ids,
    initial_ratings=issuer_rating_idx,
)

print(f'Bonds: {n_bonds}  |  Issuers: {n_issuers}  |  CDI PV: {bonds.pv0(val_date, curve, issuer_rating_idx[issuer_of]).sum():,.0f}')

## 2 — Base-case simulation

In [ ]:
rng = np.random.default_rng(123)
n_sim = 20_000

cr_sim  = cr_model.simulate(rng, n_sim, n_years)
cdi_res = mandate.run(val_date, curve, cr_sim)

print(cdi_res.summary().to_markdown(floatfmt=',.0f'))

## 3 — Visualisation

In [ ]:
# ── Plot configuration ───────────────────────────────────────────────
plt.rcParams.update({
    'figure.facecolor': '#0e1117',
    'axes.facecolor':   '#0e1117',
    'axes.edgecolor':   '#2d3139',
    'axes.labelcolor':  '#c6cdd5',
    'text.color':       '#c6cdd5',
    'xtick.color':      '#8b949e',
    'ytick.color':      '#8b949e',
    'grid.color':       '#1c2028',
    'grid.alpha':       0.7,
    'font.family':      'monospace',
    'font.size':        10,
    'axes.grid':        True,
    'figure.dpi':       130,
})

C = {
    'cyan':   '#58a6ff',
    'green':  '#3fb950',
    'orange': '#d29922',
    'red':    '#f85149',
    'purple': '#bc8cff',
    'grey':   '#484f58',
}

def fmt_millions(x, _):
    return f'{x/1e6:.0f}M' if abs(x) >= 1e6 else f'{x/1e3:.0f}k'

print('Plot theme loaded')

In [ ]:
# ── 3.1  Asset fan chart (percentile bands) ─────────────────────────
fig, ax = plt.subplots(figsize=(12, 5))
years_plot = np.arange(1, cdi_res.sched.T + 1)
assets = cdi_res.wf.assets

for lo, hi, alpha in [(5, 95, 0.12), (25, 75, 0.25)]:
    ax.fill_between(
        years_plot,
        np.percentile(assets, lo, axis=0),
        np.percentile(assets, hi, axis=0),
        color=C['cyan'], alpha=alpha, linewidth=0,
    )
ax.plot(years_plot, np.median(assets, axis=0), color=C['cyan'], lw=2, label='Median')
ax.plot(years_plot, np.percentile(assets, 1, axis=0), color=C['red'], lw=1, ls='--', label='1st pctl')
ax.plot(years_plot, cdi_res.sched.liab_pv_gaap, color=C['orange'], lw=1.5, ls=':', label='GAAP liab PV')

ax.set_title('CDI ASSET VALUE — FAN CHART', fontsize=13, fontweight='bold', loc='left')
ax.set_xlabel('Year'); ax.set_ylabel('Asset Value')
ax.yaxis.set_major_formatter(mtick.FuncFormatter(fmt_millions))
ax.legend(frameon=False, fontsize=9)
fig.tight_layout()
plt.show()

In [ ]:
# ── 3.2  GAAP funding level distribution over time ───────────────────
fl_gaap = assets / cdi_res.sched.liab_pv_gaap[np.newaxis, :]

fig, ax = plt.subplots(figsize=(12, 5))
for lo, hi, alpha in [(1, 99, 0.08), (5, 95, 0.15), (25, 75, 0.30)]:
    ax.fill_between(
        years_plot,
        np.percentile(fl_gaap, lo, axis=0),
        np.percentile(fl_gaap, hi, axis=0),
        color=C['green'], alpha=alpha, linewidth=0,
    )
ax.plot(years_plot, np.median(fl_gaap, axis=0), color=C['green'], lw=2)
ax.axhline(1.0, color=C['orange'], ls='--', lw=1, label='100% funded')

ax.set_title('GAAP FUNDING LEVEL DISTRIBUTION', fontsize=13, fontweight='bold', loc='left')
ax.set_xlabel('Year'); ax.set_ylabel('Funding Level')
ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0, decimals=0))
ax.legend(frameon=False, fontsize=9)
fig.tight_layout()
plt.show()

In [ ]:
# ── 3.3  PV(obligations) distribution ────────────────────────────────
pv_hgb  = cdi_res.pv_series('hgb_payment')
pv_perf = cdi_res.pv_series('performance_payment')
pv_fee  = cdi_res.pv_series('fee')

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax, data, title, col in zip(
    axes,
    [pv_hgb, pv_perf, pv_fee],
    ['PV(HGB Payments)', 'PV(Performance Payments)', 'PV(Fees)'],
    [C['red'], C['orange'], C['cyan']],
):
    ax.hist(data[data > 0] if data.sum() > 0 else data,
            bins=60, color=col, alpha=0.7, edgecolor='none')
    ax.axvline(np.mean(data), color='white', ls='--', lw=1)
    ax.set_title(title, fontsize=10, fontweight='bold')
    ax.xaxis.set_major_formatter(mtick.FuncFormatter(fmt_millions))
    ax.set_ylabel('')

fig.suptitle('PV OF OBLIGATIONS & FEES — BASE CASE', fontsize=13, fontweight='bold', y=1.02)
fig.tight_layout()
plt.show()

In [ ]:
# ── 3.4  Net asset return heatmap (year × percentile) ────────────────
qs = [1, 5, 10, 25, 50, 75, 90, 95, 99]
ret = cdi_res.wf.net_asset_return
hm_data = np.array([np.percentile(ret, q, axis=0) for q in qs])

fig, ax = plt.subplots(figsize=(14, 4))
im = ax.imshow(hm_data, aspect='auto', cmap='RdYlGn', vmin=-0.05, vmax=0.08)
ax.set_yticks(range(len(qs))); ax.set_yticklabels([f'{q}%' for q in qs])
ax.set_xticks(range(0, n_years, 2)); ax.set_xticklabels(range(1, n_years + 1, 2))
ax.set_xlabel('Year'); ax.set_ylabel('Percentile')
ax.set_title('NET ASSET RETURN — PERCENTILE HEATMAP', fontsize=13, fontweight='bold', loc='left')
cb = fig.colorbar(im, ax=ax, format=mtick.PercentFormatter(1.0, decimals=1), shrink=0.9)
fig.tight_layout()
plt.show()

## 4 — Stress scenarios

In [ ]:
# ── 4.1  Zero recovery ───────────────────────────────────────────────
bonds_zr = BondUniverse(
    cashflows=cf_matrix, dates=dates,
    issuer_ids=issuer_of, recoveries=np.zeros(n_bonds),
    spread_table=base_spreads,
)
mandate_zr = CDIMandate(
    bonds=bonds_zr, liabs=liabs, cfg=cfg,
    cdi_weights=cdi_w, cmbp_weights=cmbp_w,
    cash_0=2e6,
)
cdi_zr = mandate_zr.run(val_date, curve, cr_sim)
print('ZERO RECOVERY')
print(cdi_zr.summary().to_markdown(floatfmt=',.0f'))

In [ ]:
# ── 4.2  High correlation (rho_e = 0.60) ────────────────────────────
cr_model_hc = CreditModel(
    tmatrix=tmat, rho_e=0.60, rho_s=np.array([0.60]),
    n_issuers=n_issuers, sector_ids=sector_ids,
    initial_ratings=issuer_rating_idx,
)
rng_hc = np.random.default_rng(123)
cr_hc  = cr_model_hc.simulate(rng_hc, n_sim, n_years)
cdi_hc = mandate.run(val_date, curve, cr_hc)
print('HIGH CORRELATION (ρ = 0.60)')
print(cdi_hc.summary().to_markdown(floatfmt=',.0f'))

In [ ]:
# ── 4.3  Comparative fan chart ───────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharey=True)

scenarios = [
    ('BASE CASE',         cdi_res.wf.assets, C['cyan']),
    ('ZERO RECOVERY',     cdi_zr.wf.assets,  C['red']),
    ('HIGH CORRELATION',  cdi_hc.wf.assets,   C['purple']),
]

for ax, (title, data, col) in zip(axes, scenarios):
    for lo, hi, alpha in [(5, 95, 0.12), (25, 75, 0.28)]:
        ax.fill_between(
            years_plot, np.percentile(data, lo, axis=0),
            np.percentile(data, hi, axis=0),
            color=col, alpha=alpha, linewidth=0,
        )
    ax.plot(years_plot, np.median(data, axis=0), color=col, lw=2)
    ax.plot(years_plot, np.percentile(data, 1, axis=0), color=C['red'], lw=1, ls='--')
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.set_xlabel('Year')
    ax.yaxis.set_major_formatter(mtick.FuncFormatter(fmt_millions))

axes[0].set_ylabel('Asset Value')
fig.suptitle('STRESS SCENARIO COMPARISON', fontsize=14, fontweight='bold', y=1.02)
fig.tight_layout()
plt.show()

In [ ]:
# ── 4.4  Tail-risk comparison bar chart ──────────────────────────────
labels = ['Base Case', 'Zero Recovery', 'High Correlation']
results = [cdi_res, cdi_zr, cdi_hc]

metrics = pd.DataFrame({
    'Scenario': labels,
    'E[PV(HGB)]':  [r.pv_series('hgb_payment').mean() for r in results],
    'E[PV(Perf)]': [r.pv_series('performance_payment').mean() for r in results],
    'E[PV(Fee)]':  [r.pv_series('fee').mean() for r in results],
    '99.5% PV(Perf)': [np.percentile(r.pv_series('performance_payment'), 99.5) for r in results],
})

fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(labels))
w = 0.22
cols = [C['red'], C['orange'], C['cyan'], C['purple']]

for i, col_name in enumerate(['E[PV(HGB)]', 'E[PV(Perf)]', 'E[PV(Fee)]', '99.5% PV(Perf)']):
    ax.bar(x + i * w, metrics[col_name], w, color=cols[i], alpha=0.85, label=col_name)

ax.set_xticks(x + 1.5 * w); ax.set_xticklabels(labels)
ax.yaxis.set_major_formatter(mtick.FuncFormatter(fmt_millions))
ax.legend(frameon=False, fontsize=9, ncol=2)
ax.set_title('EXPECTED OBLIGATIONS BY SCENARIO', fontsize=13, fontweight='bold', loc='left')
fig.tight_layout()
plt.show()

## 5 — ICAAP (nested Monte Carlo)

In [ ]:
rng_icaap = np.random.default_rng(456)
icaap = mandate.run_icaap(
    val_date, curve, cr_model, rng_icaap,
    n_outer=100, n_inner=100, n_years=25, chunk_size=25,
)
print(icaap.summary().to_markdown(floatfmt=',.0f'))

In [ ]:
# ── 5.1  Verify chunk invariance ─────────────────────────────────────
rng_v2 = np.random.default_rng(456)
icaap_v2 = mandate.run_icaap(
    val_date, curve, cr_model, rng_v2,
    n_outer=100, n_inner=100, n_years=25, chunk_size=10,
)

diff = np.max(np.abs(icaap.pv_perf - icaap_v2.pv_perf))
print(f'Max |PV(perf)| difference between chunk_size=25 and chunk_size=10: {diff:.6f}')
assert diff < 1e-6, 'CHUNK INVARIANCE VIOLATED'

In [ ]:
# ── 5.2  ICAAP distribution plots ────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

for ax, data, title, col in zip(
    axes,
    [icaap.pv_hgb, icaap.pv_perf, icaap.pv_fee],
    ['E[PV(HGB) | t=1]', 'E[PV(Perf) | t=1]', 'E[PV(Fee) | t=1]'],
    [C['red'], C['orange'], C['cyan']],
):
    # CDF
    sorted_d = np.sort(data)
    cdf = np.arange(1, len(sorted_d) + 1) / len(sorted_d)
    ax.fill_between(sorted_d, 0, cdf, color=col, alpha=0.25)
    ax.plot(sorted_d, cdf, color=col, lw=2)

    # Percentile markers
    for q, ls in [(0.50, '-'), (0.95, '--'), (0.995, ':')]:
        v = np.percentile(data, q * 100)
        ax.axvline(v, color='white', ls=ls, lw=0.8, alpha=0.6)
        ax.text(v, 0.02, f'{q:.1%}', color='white', fontsize=7,
                ha='center', transform=ax.get_xaxis_transform())

    ax.set_title(title, fontsize=10, fontweight='bold')
    ax.xaxis.set_major_formatter(mtick.FuncFormatter(fmt_millions))
    ax.set_ylabel('CDF')
    ax.set_ylim(0, 1.05)

fig.suptitle('ICAAP — 1-YEAR CONDITIONAL PV DISTRIBUTIONS', fontsize=13, fontweight='bold', y=1.02)
fig.tight_layout()
plt.show()

In [ ]:
# ── 5.3  Risk waterfall ──────────────────────────────────────────────
def risk_metric(arr, q=99.5):
    return np.percentile(arr, q) - np.mean(arr)

rm = {
    'E[PV(Perf)]': np.mean(icaap.pv_perf),
    'Risk margin\n(99.5% – mean)': risk_metric(icaap.pv_perf),
    'E[PV(HGB)]': np.mean(icaap.pv_hgb),
    'E[PV(Fee)]': -np.mean(icaap.pv_fee),   # negative = income
}

fig, ax = plt.subplots(figsize=(8, 5))
labels_wf = list(rm.keys())
values_wf = list(rm.values())
colors_wf = [C['orange'], C['red'], C['red'], C['green']]

cumulative = 0
for i, (lbl, val, col) in enumerate(zip(labels_wf, values_wf, colors_wf)):
    ax.barh(i, val, left=cumulative, color=col, alpha=0.8, height=0.6)
    ax.text(cumulative + val / 2, i, f'{val/1e6:+.1f}M',
            ha='center', va='center', fontsize=9, fontweight='bold', color='white')
    cumulative += val

ax.set_yticks(range(len(labels_wf))); ax.set_yticklabels(labels_wf)
ax.axvline(0, color=C['grey'], lw=1)
ax.axvline(cumulative, color='white', ls='--', lw=1)
ax.text(cumulative, len(labels_wf) - 0.5, f'Net: {cumulative/1e6:+.1f}M',
        ha='center', fontsize=10, fontweight='bold', color='white')
ax.xaxis.set_major_formatter(mtick.FuncFormatter(fmt_millions))
ax.set_title('ICAAP RISK WATERFALL', fontsize=13, fontweight='bold', loc='left')
ax.invert_yaxis()
fig.tight_layout()
plt.show()

## 6 — Transition matrix diagnostics

In [ ]:
# ── 6.1  Expected time to default ────────────────────────────────────
ttd = tmat.expected_time_to_default()

fig, ax = plt.subplots(figsize=(12, 4))
x_pos = np.arange(len(ttd))
bars = ax.bar(x_pos, ttd.values, color=C['cyan'], alpha=0.8)

# Colour-code by risk band
for i, (bar, val) in enumerate(zip(bars, ttd.values)):
    if val < 50:
        bar.set_color(C['red'])
    elif val < 200:
        bar.set_color(C['orange'])

ax.set_xticks(x_pos); ax.set_xticklabels(ttd.index, rotation=45, ha='right', fontsize=8)
ax.set_ylabel('Expected Years')
ax.set_title('EXPECTED TIME TO DEFAULT BY RATING', fontsize=13, fontweight='bold', loc='left')
fig.tight_layout()
plt.show()

In [ ]:
# ── 6.2  Simulated rating migration paths (sample) ──────────────────
fig, ax = plt.subplots(figsize=(12, 5))

# Pick 50 random paths for a single BBB issuer
bbb_issuers = np.where(issuer_rating_idx == schema.to_idx(np.array(['BBB']))[0])[0]
if len(bbb_issuers) > 0:
    iss = bbb_issuers[0]
    sample_paths = cr_sim.paths[:50, :, iss]           # (50, n_years+1)
    for p in sample_paths:
        ax.plot(range(n_years + 1), p, color=C['cyan'], alpha=0.15, lw=0.8)
    ax.axhline(schema.default_idx, color=C['red'], ls='--', lw=1, label='Default')

    # Y-axis: rating labels
    n_show = schema.n
    ax.set_yticks(range(n_show))
    ax.set_yticklabels(schema.labels, fontsize=7)
    ax.invert_yaxis()
    ax.set_xlabel('Year')
    ax.set_title(f'SIMULATED RATING PATHS — ISSUER (BBB)', fontsize=13, fontweight='bold', loc='left')
    ax.legend(frameon=False, fontsize=9)

fig.tight_layout()
plt.show()